# Getting the most out of the room-finder

The room-finder is a model that looks at a floor plan and says *here is a room, and here,
and here* — all of them at once, without reading a word of the printed text. It is the
strongest thing published for this job. This notebook starts from it and does nothing but
make it better.

Everything is measured against **the room-finder as published, on the plans as published**.
Every step after that changes one thing and says whether it helped.

**Attach a GPU** in the sidebar kernel settings — a T4 is plenty. **About an hour**, most of
it one compile.

### How to run it

Top to bottom, but **you** decide what to keep. Every step draws all 25 plans before and
after the change, and is followed by a cell holding a single `True` / `False`. Set it, run it,
and that decision is what the next step builds on.

The score gets a vote and prints its opinion in words. It is not the decider: it cannot see a
room that has grown out past the outer wall, and it counts a plan read with three tidy rooms
instead of six as a partial success. The pictures show both. The defaults are all `True`, so
a straight run keeps everything — which is unlikely to be the right answer, and is why the
pictures are there.

---

### What is actually adjustable

The model's weights are fixed; nobody is training anything here. What can be changed is
**which trained model**, **what picture it is shown**, and **what is done with its answer**.
That turns out to be a lot:

| | what it changes |
|---|---|
| **the checkpoint** | five are published, trained on different collections of plans and at different resolutions. Only two have ever been tried here |
| **the picture** | it shrinks whatever it is given to a small square. A plan surrounded by white margin arrives smaller than one cropped to the drawing, and a pale wall may not survive the shrink at all |
| **the pieces** | show it quarters of the plan instead of the whole thing, so small rooms are big enough to see |
| **several views** | ask it about the plan flipped and slightly turned, and keep the rooms that turn up more than once |
| **its own answer** | drop slivers, merge duplicates, and pull the corners onto the walls |

### How it is judged

Two numbers, both on the same 25 plans, and neither can be won by finding fewer rooms:

**Rooms found.** The plan prints its own room names. A caption that lands inside exactly one
predicted room is a room the software found; one that lands in no room, or in two, is a room
it missed or split. Delete a room and this falls.

**Wall match.** Walk around each room's edge and ask, at every step, whether there is a wall
there. This says whether the outline is in the right *place*.

The score that ranks the table is the two multiplied, so a reading has to do both.

**Neither is the last word.** Every step is drawn on every plan, before and after, and you
keep or drop it yourself.

**What is not measured:** a room that has ballooned out past the outer wall. Four different
ways of finding the building's true outline have now been tried and all four failed — the
wall model does not mark thick solid outer walls, so there is no closed shape to test
against. **That is why the pictures at the end are the arbiter, not the table.**

# Part 1 — Setup

Plumbing. Run these eight cells and don't read them.

### 1.1 · Is there a GPU?

In [ ]:
import subprocess

import torch

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("no nvidia-smi on this runtime")
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU, then rerun"

### 1.2 · Get the two codebases

In [ ]:
import os
import subprocess
from pathlib import Path

# If you mounted a Modal Volume, put its path here — the results get copied there at
# the end and survive the machine being stopped. Leave it None and everything lives
# only as long as this kernel, and you download the zip from the sidebar instead.
VOLUME = None                      # e.g. Path("/mnt/my-volume")

# Everything this notebook makes lives here.
ROOT = Path("/root/plan-reading")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

R2S = ROOT / "Raster2Seq"
OURS = ROOT / "visit-it"


def clone(url, dest):
    if dest.exists():
        print(f"{dest.name}: already here")
        return
    r = subprocess.run(["git", "clone", "--depth", "1", url, str(dest)],
                       capture_output=True, text=True)
    print(f"{dest.name}: {'cloned' if not r.returncode else 'FAILED'}")
    if r.returncode:
        print(r.stderr[-600:])


clone("https://github.com/Cornell-VAILab/Raster2Seq.git", R2S)
clone("https://github.com/romainbigare/visit-it.git", OURS)
os.chdir(R2S)

### 1.3 · Install what is missing — carefully

The image ships NumPy, OpenCV and Matplotlib as one set, all built against each other.
Moving any of them breaks the rest, so NumPy is pinned where it already is and only what is
genuinely absent gets installed.

In [ ]:
import shutil
import subprocess
import sys

import numpy

# Hold NumPy exactly where the image put it. OpenCV, SciPy and Matplotlib are all
# built against a particular NumPy, and moving it breaks every one of them.
PIN = f"numpy=={numpy.__version__}"
# Read off what the code actually imports, not guessed at. The vendored copy of
# detectron2 drags in a long tail of small packages -- cloudpickle, hydra, iopath,
# tabulate, termcolor, black -- that Colab happened to ship and a clean image does
# not. The preflight in 1.6 catches anything still missing.
NEEDED = ["opencv-python-headless", "scikit-image", "scipy", "shapely", "plotly",
          "imageio", "descartes", "omegaconf", "fvcore", "pycocotools",
          "segmentation-models-pytorch", "safetensors", "pytesseract", "timm",
          "cloudpickle", "hydra-core", "iopath", "tabulate", "termcolor", "black",
          "pyyaml", "yacs", "portalocker"]


def pip_install(*packages):
    """Install with NumPy held still. Use this for everything in this notebook —
    one unpinned install anywhere is enough to break the binary packages."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIN, *packages],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise SystemExit(f"pip could not install {packages} alongside {PIN}")


print(f"holding {PIN}")
pip_install(*NEEDED)

# Tesseract is a program, not a Python package, and our own reading needs it to
# find the room names printed on the plan.
if shutil.which("tesseract") is None:
    subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "tesseract-ocr"],
                   capture_output=True)
HAVE_TESSERACT = shutil.which("tesseract") is not None
print("tesseract:", "installed" if HAVE_TESSERACT else
      "NOT AVAILABLE — step 0 will find fewer rooms than it should")

check = subprocess.run(
    [sys.executable, "-c",
     "import numpy, cv2, matplotlib, shapely, plotly, imageio, torch, skimage;"
     "print(numpy.__version__, cv2.__version__, matplotlib.__version__)"],
    capture_output=True, text=True)
if check.returncode:
    print(check.stderr[-1200:])
    raise SystemExit("something broke — restart the kernel and run from the top")
print("numpy / opencv / matplotlib:", check.stdout.strip(), "— all fine")

### 1.4 · Make the room-finder run, and build it

Raster2Seq was written for early-2024 libraries and four things have moved since. Each has an
exact modern equivalent, so these are renames, not rewrites. Then it compiles its two GPU
pieces — this is the slow cell, around five minutes.

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

import torch

REPO = R2S

CMAP_OLD = "from matplotlib.cm import get_cmap"
CMAP_MARK = "# patched: matplotlib >= 3.9"
CMAP_NEW = f"""try:
    {CMAP_OLD}
except ImportError:                      {CMAP_MARK} removed it
    from matplotlib import colormaps

    def get_cmap(name=None, lut=None):
        return colormaps[name]"""


def modernise(root: Path) -> dict:
    """Four renames. Each is guarded against matching its own output, so running
    this twice changes nothing the second time."""
    hits = {}
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix in (".cu", ".cuh", ".cpp", ".h", ".hpp"):
            src = original = path.read_text()
            src, a = re.subn(r"(AT_DISPATCH_\w+\(\s*)(\w+)\.type\(\)",
                             r"\1\2.scalar_type()", src)          # Tensor::type() is gone
            src, b = re.subn(r"(\w+)\.type\(\)\.is_cuda\(\)", r"\1.is_cuda()", src)
            n = a + b
        elif path.suffix == ".py":
            src = original = path.read_text()
            n = 0
            if CMAP_MARK not in src and CMAP_OLD in src:          # matplotlib 3.9
                src = src.replace(CMAP_OLD, CMAP_NEW, 1)
                n += 1
            src, c = re.subn(                                     # pytorch 2.6
                r"torch\.load\(([^)]*?map_location=[^)]*?)\)",
                lambda m: (m.group(0) if "weights_only" in m.group(1)
                           else f"torch.load({m.group(1)}, weights_only=False)"),
                src)
            n += c
        else:
            continue
        if src != original:
            path.write_text(src)
            hits[str(path.relative_to(root))] = n
    return hits


edits = modernise(REPO)
print(f"patched {len(edits)} files" if edits else "nothing left to patch")

major, minor = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
print(f"\nbuilding for {torch.cuda.get_device_name(0)} — about five minutes\n")


def build(where: Path) -> bool:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--no-build-isolation", "."],
                       cwd=where, capture_output=True, text=True, env=os.environ)
    print(f"{where.name}: {'built' if not r.returncode else 'FAILED'}")
    if r.returncode:
        for line in (r.stdout + r.stderr).splitlines()[-10:]:
            print("   ", line)
    return not r.returncode


build(REPO / "models" / "ops")
if not build(REPO / "diff_ras"):
    print("  (the second one is only used by a feature we switch off — this is fine)")

### 1.5 · Confirm the room-finder works

If the GPU piece did not build, there is a slower pure-Python version of the same
calculation. Identical answers, several times slower. Either is fine here.

In [ ]:
import importlib
import importlib.util
import site
import sys
from pathlib import Path

import torch

REPO = R2S
sys.path.insert(0, str(REPO))
FUNC = REPO / "models" / "ops" / "functions" / "ms_deform_attn_func.py"
UPSTREAM = "import MultiScaleDeformableAttention as MSDA"

SHIM = """
try:
    import MultiScaleDeformableAttention as MSDA
except ImportError:
    # The GPU piece did not build. The same calculation in plain PyTorch is
    # further down this file, so route through it. Inference only.
    class _PurePythonMSDA:
        @staticmethod
        def ms_deform_attn_forward(value, value_spatial_shapes, value_level_start_index,
                                   sampling_locations, attention_weights, im2col_step):
            return ms_deform_attn_core_pytorch(
                value, value_spatial_shapes, sampling_locations, attention_weights)

        @staticmethod
        def ms_deform_attn_backward(*_args, **_kwargs):
            raise RuntimeError("inference only without the compiled extension")

    MSDA = _PurePythonMSDA()
"""


def find_extension():
    """A package installed while this kernel was already running is invisible to it
    until the import caches are dropped. Worth trying before concluding it failed."""
    importlib.invalidate_caches()
    try:
        return importlib.import_module("MultiScaleDeformableAttention")
    except ImportError:
        pass
    site.main()
    roots = list(site.getsitepackages())
    for root in roots:
        for egg in Path(root).glob("MultiScaleDeformableAttention*.egg"):
            if str(egg) not in sys.path:
                sys.path.insert(0, str(egg))
    importlib.invalidate_caches()
    try:
        return importlib.import_module("MultiScaleDeformableAttention")
    except ImportError:
        return None


if find_extension() is not None:
    SLOW_PATH = False
    print("room-finder: fast GPU version")
else:
    src = FUNC.read_text()
    if "_PurePythonMSDA" not in src:
        FUNC.write_text(src.replace(UPSTREAM, SHIM.strip(), 1))
    for name in [m for m in sys.modules if "ms_deform_attn" in m]:
        del sys.modules[name]
    SLOW_PATH = True
    print("room-finder: slower plain-PyTorch version (same answers)")

spec = importlib.util.spec_from_file_location("_check", FUNC)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
shapes = torch.as_tensor([[8, 6], [4, 3]], dtype=torch.long, device="cuda")
sizes = shapes[:, 0] * shapes[:, 1]
with torch.no_grad():
    out = mod.MSDeformAttnFunction.apply(
        torch.randn(2, int(sizes.sum()), 8, 32, device="cuda"), shapes,
        torch.cat([sizes.new_zeros(1), sizes.cumsum(0)[:-1]]),
        torch.rand(2, 7, 8, 2, 4, 2, device="cuda"),
        torch.rand(2, 7, 8, 2, 4, device="cuda"), 64)
assert tuple(out.shape) == (2, 7, 256) and torch.isfinite(out).all()
print("   checked — it computes the right thing")

# The model's own import chain, exercised before we spend five minutes discovering
# it at the first run. A missing module names itself in the error, so install it and
# try again rather than making you paste the traceback back to me.
PIP_NAME = {"cv2": "opencv-python-headless", "skimage": "scikit-image",
            "PIL": "pillow", "sklearn": "scikit-learn", "yaml": "pyyaml",
            "hydra": "hydra-core", "shapely": "shapely", "pycocotools": "pycocotools",
            "google": "protobuf", "cairosvg": "cairosvg", "drawsvg": "drawsvg",
            "svgpathtools": "svgpathtools", "svgwrite": "svgwrite",
            "plyfile": "plyfile", "webcolors": "webcolors"}


def preflight(rounds=12):
    tried = set()
    for _ in range(rounds):
        r = subprocess.run([sys.executable, "-c", "import predict"],
                           cwd=str(R2S), capture_output=True, text=True)
        if r.returncode == 0:
            return True
        err = r.stderr.strip().splitlines()
        missing = next((ln.split("'")[1] for ln in reversed(err)
                        if "ModuleNotFoundError: No module named" in ln), None)
        if not missing:
            print("   the model cannot start, and it is not a missing package:")
            print("\n".join(err[-12:]))
            return False
        pkg = PIP_NAME.get(missing, missing)
        if missing in tried:
            # Installing it did not make it importable, so the name we guessed is
            # wrong. Say which, rather than reinstalling it until the loop runs out.
            print(f"   installed {pkg}, but '{missing}' still will not import.")
            print(f"   Add the right pip name for '{missing}' to PIP_NAME above "
                  f"and re-run this cell.")
            return False
        tried.add(missing)
        print(f"   missing {missing} → installing {pkg}")
        try:
            pip_install(pkg)
        except SystemExit:
            print(f"   could not install {pkg} — stopping here")
            return False
    print(f"   still missing things after {rounds} rounds — stopping")
    return False


print()
print("checking the model can start")
READY = preflight()
print("   ready" if READY else "   NOT ready — the steps below will report nothing")

### 1.6 · The 25 test plans

Downloaded straight from the addresses recorded in our own test set, so nothing needs
uploading and nothing needs to exist on your machine.

In [ ]:
import json
import ssl
import urllib.request
from pathlib import Path

PLANS = ROOT / "plans" / "original"
PLANS.mkdir(parents=True, exist_ok=True)
golden = json.loads((OURS / "data" / "golden" / "golden_set.json").read_text())

ctx = ssl.create_default_context()
missing = []
for listing in golden["listings"]:
    plans = listing.get("floorplans") or []
    if not plans:
        missing.append(listing["listing_id"])
        continue
    dest = PLANS / f"{listing['listing_id']}.png"
    if dest.exists():
        continue
    try:
        req = urllib.request.Request(plans[0]["url"], headers={"User-Agent": "Mozilla/5.0"})
        dest.write_bytes(urllib.request.urlopen(req, context=ctx, timeout=60).read())
    except Exception as exc:
        print("  could not fetch", listing["listing_id"], exc)
        missing.append(listing["listing_id"])

IDS = sorted(p.stem for p in PLANS.glob("*.png"))
print(f"{len(IDS)} plans ready" + (f"  ·  {len(missing)} listings have no plan" if missing else ""))

### 1.7 · The scorecard

Two numbers per plan, defined in the cell below. Read them if you like; you do not need to.

In [ ]:
import sys
import urllib.request
import warnings
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

sys.path.insert(0, str(OURS))
warnings.filterwarnings("ignore")

from pipeline.floorplan import ocr as ocr_mod
from pipeline.floorplan import preprocess, vectorise, wallnet

(OURS / "models").mkdir(parents=True, exist_ok=True)
wallnet.MODEL_PATH = OURS / "models" / "plan_walls.safetensors"
if not wallnet.MODEL_PATH.exists():
    urllib.request.urlretrieve(wallnet.MODEL_URL, wallnet.MODEL_PATH)
assert wallnet.available(), "the wall model did not load"

# Two things per plan, worked out once: a map of which pixels are wall, and where
# the plan prints its room names. Everything below is scored against these, and
# both live in the plan's own pixels -- so does every answer, whatever picture the
# model was actually shown.
import hashlib

_READ = {}


def text_on(rgb):
    """What the plan says, read once per picture and then remembered.

    Reading a plan takes ten to thirty seconds, and the steps below show the model
    the same picture again and again -- once per step, four more times for the four
    views. Without this the run spends longer reading text it has already read than
    it does on the GPU.
    """
    key = hashlib.blake2b(np.ascontiguousarray(rgb).tobytes(), digest_size=16).digest()
    if key not in _READ:
        _READ[key] = ocr_mod.read(rgb)
    return _READ[key]


WALLS, CAPTIONS = {}, {}
print(f"reading {len(IDS)} plans (a few minutes)")
for lid in IDS:
    rgb = np.array(Image.open(PLANS / f"{lid}.png").convert("RGB"))
    ink, _ = preprocess.ink_mask(rgb)
    WALLS[lid] = wallnet.barrier(rgb, ink, [])
    CAPTIONS[lid] = [(float(b.cx), float(b.cy), b.label)
                     for b in text_on(rgb).blocks if b.is_room_caption]
print(f"done -- the plans print {sum(len(c) for c in CAPTIONS.values())} room names "
      f"between them")


def rooms_found(polys, captions):
    """Share of the printed room names that land inside exactly one room.

    Inside none means the room was missed. Inside two means it was split, or two
    predictions sit on top of each other. Only exactly one counts, which is why
    this cannot be improved by finding fewer rooms.
    """
    if not captions:
        return None
    hits = 0
    for cx, cy, _label in captions:
        n = sum(1 for p in polys
                if cv2.pointPolygonTest(np.round(np.asarray(p, np.float32)), (cx, cy), False) >= 0)
        hits += (n == 1)
    return hits / len(captions)


LADDER = []


def score(name, reading, note):
    """Measure a reading, put it on the ladder, and say how it did."""
    per_plan = {}
    for lid, rec in reading.items():
        polys = [r["polygon_px"] for r in rec["rooms"] if len(r["polygon_px"]) >= 3]
        wall = WALLS.get(lid)
        if not polys or wall is None or not wall.any():
            continue
        found = rooms_found(polys, CAPTIONS.get(lid, []))
        edge = wallnet.outline_on_wall(polys, wall)
        if found is None or not edge:
            continue
        edge = float(np.median(edge))
        per_plan[lid] = {"found": found, "edge": edge, "score": found * edge,
                         "rooms": len(polys)}
    if not per_plan:
        print(f"  {name}: produced nothing to score")
        return None
    med = lambda k: float(np.median([v[k] for v in per_plan.values()]))
    entry = {"name": name, "note": note, "reading": reading, "per_plan": per_plan,
             "score": med("score"), "found": med("found"), "edge": med("edge"),
             "rooms": sum(len(r["rooms"]) for r in reading.values()),
             "plans": len(per_plan)}
    LADDER.append(entry)
    print(f"  {name}")
    print(f"  score {entry['score']:.0%}   ·  rooms found {entry['found']:.0%}   ·  "
          f"wall match {entry['edge']:.0%}   ·  {entry['rooms']} rooms")
    if len(LADDER) > 1:
        base = LADDER[0]["score"]
        best = max(e["score"] for e in LADDER[:-1])
        delta = entry["score"] - base
        print(f"  against the base: {delta:+.0%}" +
              ("   ·  best so far" if entry["score"] > best + 0.005 else ""))
    return entry

### 1.8 · What the model is shown, and how its answer gets back

The model does not see the plan. It sees a small square copy of whatever picture it is
handed, with grey bars where the shape does not fit. So the picture is the main thing there
is to tune, and every step below is a change to it.

Each change also has to be undone on the way back, or the rooms come back in the wrong
place. That bookkeeping is what this cell is for.

In [ ]:
import shutil
import subprocess
import time

HUB = "https://huggingface.co/haopt/Raster2Seq/resolve/main"

# What the room types are called, per training set.
CC5K_NAMES = {0: "Outdoor", 1: "Kitchen", 2: "Living Room", 3: "Bed Room", 4: "Bath",
              5: "Entry", 6: "Storage", 7: "Garage", 8: "Undefined"}
CC5K_APERTURES = {9, 10}                       # window and door, kept out of the rooms
R2G_NAMES = {0: "unknown", 1: "living room", 2: "kitchen", 3: "bedroom", 4: "bathroom",
             5: "restroom", 6: "balcony", 7: "closet", 8: "corridor",
             9: "washing room", 10: "PS", 11: "outside"}
S3D_NAMES = {0: "living room", 1: "kitchen", 2: "bedroom", 3: "bathroom", 4: "balcony",
             5: "corridor", 6: "dining room", 7: "study", 8: "studio", 9: "store room",
             10: "garden", 11: "laundry room", 12: "office", 13: "basement",
             14: "garage", 15: "undefined"}
_CONFIGS = {}


def hub_subfolder(key):
    """Where that checkpoint lives on the hub -- read from the repo's own table,
    because the name and the folder are not always the same."""
    try:
        sys.path.insert(0, str(R2S))
        import raster2seq_hub
        canonical = raster2seq_hub.normalize_checkpoint_name(key)
        return canonical, str(raster2seq_hub.CHECKPOINTS[canonical]["subfolder"])
    except Exception:
        return key, key


def checkpoint_config(key):
    """The settings that checkpoint was trained with, straight from the hub. Never
    hand-write these: the bin count, the class count and the working resolution all
    differ between them, and getting one wrong means it will not load at all."""
    if key not in _CONFIGS:
        canonical, folder = hub_subfolder(key)
        with urllib.request.urlopen(f"{HUB}/{folder}/config.json", timeout=60) as r:
            _CONFIGS[key] = json.load(r)
        _CONFIGS[key]["_alias"] = canonical
    return _CONFIGS[key]


def flags_from(config):
    args = dict(config["inference_args"])
    for drop in ("dataset_root", "eval_set", "output_dir"):
        args.pop(drop, None)
    out = []
    for k, v in args.items():
        if v is True:
            out.append(f"--{k}")
        elif v is False or v is None:
            continue
        else:
            out += [f"--{k}", str(v)]
    return out, int(args.get("image_size", 256)), args.get("dataset_name", "cubicasa")


def undo_letterbox(poly, src_w, src_h, size):
    """The model works on a small square copy with bars down the sides. Put the
    coordinates back on the picture it was given."""
    scale = min(size / src_h, size / src_w)
    new_h, new_w = int(src_h * scale), int(src_w * scale)
    left, top = (size - new_w) // 2, (size - new_h) // 2
    p = np.asarray(poly, dtype=float).reshape(-1, 2)
    return np.stack([(p[:, 0] - left) / scale, (p[:, 1] - top) / scale], axis=1)


# ---- the changes we can make to the picture -----------------------------------
# Each returns the new picture and a function that puts a polygon found on it back
# on the plan.

def clean(rgb):
    """Page to white, ink to black, every printed word painted out. It learned on
    line drawings with no text; ours are tinted, filled and covered in writing."""
    return wallnet.blank_text(wallnet.whiten(rgb), text_on(rgb).words), (lambda p: p)


def crop(rgb):
    """Trim the margin. The model shrinks whatever it is given to one small square,
    so margin is resolution thrown away before it has looked at anything."""
    ink, _ = preprocess.ink_mask(rgb)
    out, (x0, y0) = wallnet.crop_to_drawing(rgb, ink)
    return out, (lambda p, dx=x0, dy=y0: (np.asarray(p, float) + [dx, dy]).tolist())


def pure_lines(rgb):
    """Hard black on white: grey fills and tints go, drawn lines stay.

    With the cut too high this backfires badly. A plan whose rooms are filled
    mid-grey comes back with the whole flat solid black -- every room gone, not
    just its tint. So the cut is low, and if the result is still more than a third
    ink the plan is left alone.
    """
    lum = cv2.cvtColor(wallnet.whiten(rgb), cv2.COLOR_RGB2GRAY)
    bw = np.where(lum < 160, 0, 255).astype(np.uint8)
    if (bw == 0).mean() > 0.35:
        return rgb, (lambda p: p)
    return cv2.cvtColor(bw, cv2.COLOR_GRAY2RGB), (lambda p: p)


def thicker(rgb):
    """Fatten every stroke enough to survive the shrink.

    A wall drawn two pixels wide on a 1400-pixel plan is a third of a pixel once the
    model has resized it to 256, which is to say gone. Grow the dark strokes until
    they are at least one pixel wide after that resize.
    """
    need = max(1, int(round(rgb.shape[1] / 256.0 / 2)))
    dark = cv2.erode(cv2.cvtColor(wallnet.whiten(rgb), cv2.COLOR_RGB2GRAY),
                     np.ones((2 * need + 1,) * 2, np.uint8))
    return cv2.cvtColor(dark, cv2.COLOR_GRAY2RGB), (lambda p: p)


def show(steps, tag):
    """Write the picture every plan turns into, and keep the way back."""
    where = ROOT / "shown" / tag
    if where.exists():
        shutil.rmtree(where)
    where.mkdir(parents=True)
    back = {}
    for lid in IDS:
        rgb = np.array(Image.open(PLANS / f"{lid}.png").convert("RGB"))
        undo = []
        for step in steps:
            rgb, f = step(rgb)
            undo.append(f)
        Image.fromarray(rgb).save(where / f"{lid}.png")
        back[lid] = undo[::-1]              # applied in reverse on the way back
    return where, back


def onto_plan(poly, undo):
    for f in undo:
        poly = f(poly)
    return np.asarray(poly, float).tolist()


def predict(images_dir, tag, checkpoint):
    """Run the model over a directory of pictures. Rooms come back in the pixels of
    the picture that was handed over -- not the plan's."""
    if not READY:
        print("  skipped: the room-finder could not start -- see the preflight in 1.5")
        return {}
    try:
        config = checkpoint_config(checkpoint)
    except Exception as exc:
        print(f"  could not read the settings for '{checkpoint}': {exc}")
        return {}
    flags, size, dataset = flags_from(config)
    names = {"r2g": R2G_NAMES, "stru3d": S3D_NAMES}.get(dataset, CC5K_NAMES)
    apertures = CC5K_APERTURES if names is CC5K_NAMES else set()

    out_dir = ROOT / "runs" / tag
    if out_dir.exists():
        shutil.rmtree(out_dir)
    t0 = time.time()
    r = subprocess.run(["python", "predict.py", f"--dataset_root={images_dir}",
                        f"--output_dir={out_dir}",
                        f"--checkpoint=hf:{config['_alias']}", *flags],
                       cwd=str(R2S), capture_output=True, text=True)
    if r.returncode:
        print(f"  the model failed on '{tag}':")
        for line in (r.stdout + r.stderr).splitlines()[-8:]:
            print("     ", line)
        return {}

    reading = {}
    for jf in sorted(out_dir.rglob("jsons/*.json")):
        lid = jf.stem
        shown = Path(images_dir) / f"{lid}.png"
        src = Image.open(shown if shown.exists() else PLANS / f"{lid}.png")
        rooms = []
        for inst in json.loads(jf.read_text()):
            cid = inst["category_id"]
            if cid in apertures or cid not in names:
                continue
            rooms.append({"polygon_px": undo_letterbox(inst["segmentation"], src.width,
                                                       src.height, size).tolist(),
                          "label": names[cid]})
        reading[lid] = {"rooms": rooms}
    print(f"  {config['name']} · {size}px · {time.time() - t0:.0f}s", end="   ")
    return reading


def run_reading(steps=(), checkpoint=None, tag="run"):
    """Show the model a picture, run it, and put the answers back on the plan."""
    where, back = show(steps, tag)
    raw = predict(where, tag, checkpoint or CHECKPOINT)
    return {lid: {"rooms": [{**r, "polygon_px": onto_plan(r["polygon_px"], back[lid])}
                            for r in rec["rooms"]]}
            for lid, rec in raw.items() if lid in back}


def read(name, note, steps=(), checkpoint=None, tag=None):
    """Run it and score it in one go, for the steps that are not up for a decision."""
    tag = tag or "".join(ch if ch.isalnum() else "_" for ch in name.lower()).strip("_")[:40]
    return score(name, run_reading(steps, checkpoint, tag), note)


CHECKPOINT = "cubicasa5k"      # step 1 asks you to choose
RECIPE = []                    # grows with the steps you decide to keep
CURRENT = None                 # the reading we are building on right now

### 1.9 · Drawing the answers

Every comparison below is drawn by the same function, so the pictures are always
like-for-like: the plan on the left, then one panel per reading, each captioned with that
plan's own numbers.

In [ ]:
import colorsys

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPoly

PLAN_IMG = {lid: np.array(Image.open(PLANS / f"{lid}.png").convert("RGB")) for lid in IDS}


def hue(i):
    return colorsys.hsv_to_rgb((i * 0.61803) % 1.0, 0.55, 0.95)


def draw(ax, img, rooms, title):
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(title, fontsize=9)
    for i, room in enumerate(rooms):
        p = np.asarray(room["polygon_px"])
        if len(p) < 3:
            continue
        c = hue(i)
        ax.add_patch(MplPoly(p, closed=True, facecolor=c + (0.35,), edgecolor=c,
                             linewidth=2))
        if room.get("label"):
            ax.text(*p.mean(axis=0), room["label"], ha="center", va="center",
                    fontsize=7, weight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))


def compare(entries, labels=None, limit=None, per_row=3):
    """One block per plan: the plan itself, then what each reading made of it.

    Laid out three across rather than all in a line. A notebook scales a figure to
    the width of the cell, so six panels side by side arrive too small to judge --
    which defeats the point of drawing them.

    The caption under each panel is that plan's own numbers, not the median, so a
    reading that is excellent on twenty plans and hopeless on five shows it here
    rather than hiding inside an average.
    """
    entries = list(entries)
    labels = labels or [(e or {}).get("name", "-") for e in entries]
    panels = 1 + len(entries)
    rows = -(-panels // per_row)
    for lid in (IDS if limit is None else IDS[:limit]):
        fig, axes = plt.subplots(rows, per_row, figsize=(6.2 * per_row, 6.6 * rows),
                                 squeeze=False)
        flat = axes.ravel()
        flat[0].imshow(PLAN_IMG[lid])
        flat[0].set_title(f"{lid} - the plan", fontsize=10)
        for ax, entry, label in zip(flat[1:], entries, labels):
            rec = (entry or {}).get("reading", {}).get(lid) or {}
            mine = (entry or {}).get("per_plan", {}).get(lid)
            caption = f"{len(rec.get('rooms', []))} rooms"
            if mine:
                caption += f" - found {mine['found']:.0%} - walls {mine['edge']:.0%}"
            draw(ax, PLAN_IMG[lid], rec.get("rooms", []), f"{label}\n{caption}")
        for ax in flat[panels:]:
            ax.set_visible(False)
        for ax in flat:
            ax.axis("off")
        plt.tight_layout()
        plt.show()

### 1.10 · Keeping a step, or not

Each step below runs, scores itself, and draws every plan before and after. Then it stops.
The cell after it is a plain `True` / `False` that you set: keep the change and everything
after it builds on the new picture, or drop it and nothing moves.

The score gets a vote, printed in words. It is not the decider — it cannot see a room that
has ballooned out past the outer wall, and you can.

In [ ]:
def offer(name, note, reading, step=None):
    """Measure a candidate, show it beside what we already have, and stop there.

    Nothing is adopted by this function. The score gets a vote and says so; the
    decision is the toggle in the cell that follows.
    """
    entry = score(name, reading, note)
    if entry is None:
        return None
    entry["step"] = step
    gap = entry["score"] - CURRENT["score"]
    word = "better" if gap > 0.005 else ("worse" if gap < -0.005 else "no different")
    print(f"\n  the score says {word} ({gap:+.0%}). Now look at the plans below, then set")
    print("  the toggle in the next cell. The score cannot see a room that has grown")
    print("  out past the outer wall; you can.")
    compare([CURRENT, entry], labels=["what we have", "with this change"])
    return entry


def try_step(name, note, step):
    """A change to the picture the model is shown."""
    tag = "".join(ch if ch.isalnum() else "_" for ch in name.lower()).strip("_")[:40]
    return offer(name, note, run_reading(RECIPE + [step], tag=tag), step=step)


def decide(entry, keep):
    """Your call. Keep it and everything after builds on it; drop it and nothing changes."""
    global CURRENT, RECIPE
    if entry is None:
        print("there is nothing to decide -- that step produced no reading")
        return
    was = CURRENT["score"] if CURRENT else 0.0
    if not keep:
        print(f"DROPPED: {entry['name']}")
    else:
        if entry.get("step") is not None:
            RECIPE = RECIPE + [entry["step"]]
        CURRENT = entry
        print(f"KEPT: {entry['name']}")
        if entry["score"] < was - 0.005:
            print("  (the score disagreed -- going with your call)")
    shown = " -> ".join(f.__name__ for f in RECIPE) or "the plan as published"
    print(f"  checkpoint: {CHECKPOINT}")
    print(f"  picture:    {shown}")
    print(f"  building on: {CURRENT['name']}  ({CURRENT['score']:.0%})")

---

# Part 2 — Tuning the room-finder

Each step changes one thing and is measured against the base. A step that helps is kept and
everything after it builds on it; a step that does not is dropped and said so.

## Step 0 · The base

The model as published, on the plans as published. Every number below is against this one.

In [ ]:
CURRENT = read("Base: as published",
               "the room-finder, untouched, on the original files")

## Step 1 · Which of the five trained models

The authors publish five, trained on different collections of plans and at different
resolutions. Two have ever been tried here. Their scores on *their own* test sets range from
88.7 to 99.6, but those are different test sets — the only thing that matters is which one
reads **our** plans best.

The table below gives one number each. The pictures after it give all five side by side on
every plan, which is the part worth actually looking at: a model that scores well by finding
three tidy rooms in a six-room flat looks fine in a table and obviously wrong in a picture.

In [ ]:
FIVE = ["cubicasa5k",           # real estate agents' plans, 256px
        "raster2graph",         # a different collection, 256px
        "raster2graph-512",     # the same, at twice the resolution
        "s3d-bw",               # synthetic, clean line drawings
        "s3d-density"]          # synthetic, trained on density maps rather than drawings

for key in FIVE:
    read(f"checkpoint: {key}", f"the {key} weights, everything else unchanged",
         checkpoint=key, tag=f"ckpt_{key}")

tried = [e for e in LADDER if e["name"].startswith("checkpoint: ")]
if tried:
    pick = max(tried, key=lambda e: e["score"])
    print(f"\nthe score would pick: {pick['name'].split(': ', 1)[1]}  ({pick['score']:.0%})")
    print("look at the pictures below before agreeing with it")

In [ ]:
# All five, on every plan. This is the cell to scroll through.
BY_CHECKPOINT = {e["name"].split(": ", 1)[1]: e for e in LADDER
                 if e["name"].startswith("checkpoint: ")}
compare(list(BY_CHECKPOINT.values()), labels=list(BY_CHECKPOINT))

**Which one?** Set it in the next cell.

In [ ]:
# ---------------------------------------------------------------------------
# YOUR CHOICE. The score's pick is printed above; change this line if the
# pictures say otherwise. Everything below uses whichever you name here.
# ---------------------------------------------------------------------------
CHECKPOINT = "cubicasa5k"        # cubicasa5k | raster2graph | raster2graph-512
                                 # s3d-bw | s3d-density

decide(BY_CHECKPOINT[CHECKPOINT], True)

## Step 2 · Give it a cleaner picture

It learned on plans drawn as black lines on white paper with no writing on them. Ours are
tinted, colour-filled and covered in text. This levels the page to white and paints out every
printed word.

In [ ]:
candidate = try_step("+ cleaned up",
                     "page levelled to white, every printed word painted out",
                     clean)

In [ ]:
# ---------------------------------------------------------------------------
# YOUR CALL: keep this change, or drop it?
# ---------------------------------------------------------------------------
KEEP_CLEANED = True

decide(candidate, KEEP_CLEANED)

## Step 3 · Crop to the drawing

The model shrinks whatever it is handed into one small square. A plan sitting in a wide white
margin therefore arrives smaller than the same plan cropped to its own drawing — the margin
costs resolution before the model has looked at anything.

In [ ]:
candidate = try_step("+ cropped to the drawing",
                     "margin trimmed so the plan fills the square",
                     crop)

In [ ]:
# ---------------------------------------------------------------------------
# YOUR CALL: keep this change, or drop it?
# ---------------------------------------------------------------------------
KEEP_CROPPED = True

decide(candidate, KEEP_CROPPED)

## Step 4 · Pure black on white

Grey fills, hatching and colour tints are all things it never saw in training. This throws
them away and keeps only the drawn lines.

In [ ]:
candidate = try_step("+ pure black on white",
                     "grey fills and tints removed, drawn lines kept",
                     pure_lines)

In [ ]:
# ---------------------------------------------------------------------------
# YOUR CALL: keep this change, or drop it?
# ---------------------------------------------------------------------------
KEEP_PURE = True

decide(candidate, KEEP_PURE)

## Step 5 · Fatten the strokes

A wall drawn two pixels wide on a 1400-pixel plan is a third of a pixel once the model has
resized it to 256 — which is to say, gone. This grows every dark stroke until it is at least
one pixel wide *after* that resize.

In [ ]:
candidate = try_step("+ thicker strokes",
                     "every line fattened enough to survive the shrink",
                     thicker)

In [ ]:
# ---------------------------------------------------------------------------
# YOUR CALL: keep this change, or drop it?
# ---------------------------------------------------------------------------
KEEP_THICKER = True

decide(candidate, KEEP_THICKER)

## Step 6 · Show it the plan in pieces

Same idea from the other end: cut the plan into four overlapping quarters and run the model
on each. A cupboard that was six pixels across in the whole-plan square is twelve in a
quarter.

In [ ]:
from itertools import product

from shapely.geometry import Polygon


def area_of(poly):
    p = np.asarray(poly, float)
    return abs(float(np.dot(p[:, 0], np.roll(p[:, 1], 1)) -
                     np.dot(p[:, 1], np.roll(p[:, 0], 1))) / 2)


def dedupe(rooms, overlap=0.5):
    """One room found twice is one room. Keep the bigger version."""
    kept, shapes = [], []
    for room in sorted(rooms, key=lambda r: -area_of(r["polygon_px"])):
        try:
            g = Polygon(room["polygon_px"]).buffer(0)
        except Exception:                                        # noqa: BLE001
            continue
        if g.is_empty or g.area <= 0:
            continue
        if any(g.intersection(k).area > overlap * min(g.area, k.area) for k in shapes):
            continue
        shapes.append(g)
        kept.append(room)
    return kept


def in_pieces(overlap=0.25):
    base_dir, back = show(RECIPE, "recipe")
    tiles = ROOT / "shown" / "pieces"
    if tiles.exists():
        shutil.rmtree(tiles)
    tiles.mkdir(parents=True)
    boxes = {}
    for lid in IDS:
        im = Image.open(base_dir / f"{lid}.png").convert("RGB")
        w, h = im.size
        tw, th = int(w * (0.5 + overlap / 2)), int(h * (0.5 + overlap / 2))
        for i, (cx, cy) in enumerate(product((0, w - tw), (0, h - th))):
            im.crop((cx, cy, cx + tw, cy + th)).save(tiles / f"{lid}__{i}.png")
            boxes[f"{lid}__{i}"] = (lid, cx, cy)

    raw = predict(tiles, "pieces", CHECKPOINT)
    merged = {}
    for name, rec in raw.items():
        if name not in boxes:
            continue
        lid, cx, cy = boxes[name]
        for room in rec["rooms"]:
            p = np.asarray(room["polygon_px"], float) + [cx, cy]
            merged.setdefault(lid, []).append({**room,
                                               "polygon_px": onto_plan(p, back[lid])})
    return {lid: {"rooms": dedupe(rooms)} for lid, rooms in merged.items()}


candidate = offer("+ shown in four pieces",
                  "four overlapping quarters, answers merged",
                  in_pieces())

In [ ]:
# ---------------------------------------------------------------------------
# YOUR CALL: keep this change, or drop it?
# ---------------------------------------------------------------------------
KEEP_PIECES = True

decide(candidate, KEEP_PIECES)

## Step 7 · Ask it several ways and keep what agrees

Show it the picture as it is, mirrored left-to-right, mirrored top-to-bottom, and turned two
degrees. A mirror image of a floor plan is still a floor plan, so all four answers should
agree — where they do not, one of them is wrong. Keep every room, merge the duplicates.

In [ ]:
def mirror(rgb, axis):
    h, w = rgb.shape[:2]
    if axis == "x":
        return rgb[:, ::-1].copy(), (lambda p, W=w: (np.asarray(p, float) * [-1, 1]
                                                     + [W - 1, 0]).tolist())
    return rgb[::-1].copy(), (lambda p, H=h: (np.asarray(p, float) * [1, -1]
                                              + [0, H - 1]).tolist())


def tilt(rgb, degrees=2):
    """Turned in place, so the frame does not change size -- and turned back after."""
    h, w = rgb.shape[:2]
    m = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), degrees, 1.0)
    out = cv2.warpAffine(rgb, m, (w, h), flags=cv2.INTER_LINEAR,
                         borderValue=(255, 255, 255))
    inv = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), -degrees, 1.0)
    return out, (lambda p, M=inv: (np.hstack([np.asarray(p, float),
                                              np.ones((len(p), 1))]) @ M.T).tolist())


VIEWS = {"as it is": [],
         "mirrored left-right": [lambda r: mirror(r, "x")],
         "mirrored top-bottom": [lambda r: mirror(r, "y")],
         "turned two degrees": [tilt]}

votes = {}
for view, extra in VIEWS.items():
    where, back = show(RECIPE + extra, "view_" + view.replace(" ", "_"))
    raw = predict(where, "view_" + view.replace(" ", "_"), CHECKPOINT)
    print(f"· {view}")
    for lid, rec in raw.items():
        if lid not in back:
            continue
        for room in rec["rooms"]:
            votes.setdefault(lid, []).append(
                {**room, "polygon_px": onto_plan(room["polygon_px"], back[lid])})

several = {lid: {"rooms": dedupe(rooms)} for lid, rooms in votes.items() if rooms}
candidate = offer("+ asked four ways",
                  "as it is, both mirrors and a two-degree turn, merged", several)

In [ ]:
# ---------------------------------------------------------------------------
# YOUR CALL: keep this change, or drop it?
# ---------------------------------------------------------------------------
KEEP_FOUR_WAYS = True

decide(candidate, KEEP_FOUR_WAYS)

## Step 8 · Tidy its own answer

Nothing new is run here. Take whichever reading above scored best and throw away the pieces
that cannot be rooms: slivers far smaller than anything else on the plan, and rooms sitting
on top of one another.

In [ ]:
def tidy(reading, smallest=0.06):
    """Drop slivers and overlaps. `smallest` is a fraction of that plan's median room."""
    out = {}
    for lid, rec in reading.items():
        rooms = [r for r in rec["rooms"] if len(r["polygon_px"]) >= 3]
        if not rooms:
            continue
        areas = [area_of(r["polygon_px"]) for r in rooms]
        floor = smallest * float(np.median(areas))
        keep = [r for r, a in zip(rooms, areas) if a >= floor]
        out[lid] = {"rooms": dedupe(keep, overlap=0.35)}
    return out


candidate = offer("+ tidied up", f"slivers and overlaps removed from "
                  f"'{CURRENT['name']}'", tidy(CURRENT["reading"]))

In [ ]:
# ---------------------------------------------------------------------------
# YOUR CALL: keep this change, or drop it?
# ---------------------------------------------------------------------------
KEEP_TIDIED = True

decide(candidate, KEEP_TIDIED)

---

# Part 3 — One thing done afterwards

Everything above is the room-finder. This is the one step that is not: its rooms are the
right rooms, but their corners land on a coarse grid — about 30 cm on a real flat — so the
outlines float near the walls rather than on them.

So take its rooms as they are, and let each one grow outwards until it meets a wall on the
wall model's map. Nothing is added and nothing is removed: the same rooms, with their corners
put where the walls are. Then the names printed on the plan are attached to whichever room
they were printed inside, which is something the room-finder cannot do at all — it never
reads the text.

In [ ]:
def onto_source(polys, pi):
    """Undo the straighten-and-shrink, so these come back in the plan's own pixels
    and get scored exactly like everything else above."""
    h, w = pi.ink.shape
    inv = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), -pi.deskew_deg, 1.0)
    scale = pi.scale_from_source or 1.0
    out = []
    for p in polys:
        q = np.asarray(p, float).reshape(-1, 2)
        if abs(pi.deskew_deg) > 1e-6:
            q = np.hstack([q, np.ones((len(q), 1))]) @ inv.T
        out.append((q / scale).tolist())
    return out


def onto_walls(reading, note):
    grown = {}
    for lid, rec in reading.items():
        rooms = [r for r in rec["rooms"] if len(r["polygon_px"]) >= 3]
        if not rooms:
            continue
        pi = preprocess.prepare(PLANS / f"{lid}.png")
        text = text_on(pi.rgb)
        walls = wallnet.barrier(pi.rgb, pi.ink, text.words, pi.wall_half_px)
        if walls is None or not walls.any():
            continue
        seeds = vectorise.roomfinder.to_geometry([r["polygon_px"] for r in rooms], pi)
        try:
            v = vectorise.segment_from_room_seeds(
                pi, text, seeds, mask=walls,
                fallback_labels=[str(r.get("label") or "").lower() for r in rooms])
        except Exception as exc:                                 # noqa: BLE001
            print(f"  {lid}: {type(exc).__name__}: {exc}")
            continue
        if v is None or not v.rooms:
            continue
        back = onto_source([r.polygon_px for r in v.rooms], pi)
        grown[lid] = {"rooms": [{"polygon_px": p, "label": r.label or "",
                                 "seeded_by": r.seeded_by}
                                for p, r in zip(back, v.rooms)]}
    entry = score("+ corners pulled onto the walls", grown, note)
    if entry:
        named = sum(1 for rec in grown.values() for r in rec["rooms"]
                    if r["seeded_by"] == "caption")
        print(f"  {named} of {entry['rooms']} rooms also got their name from the plan")
        entry["reading"] = grown
    return entry


print(f"growing whatever you kept: {CURRENT['name']}")
GROWN = onto_walls(CURRENT["reading"], f"'{CURRENT['name']}', grown to the walls")

### For reference only: what the software did before any of this

Not a step and not the baseline — just the line to beat, so the table shows whether the
tuned room-finder is worth switching to.

In [ ]:
ours = {}
for lid in IDS:
    try:
        pi = preprocess.prepare(PLANS / f"{lid}.png")
        text = text_on(pi.rgb)
        v = vectorise.segment(pi, text)
    except Exception as exc:                                     # noqa: BLE001
        print(f"  {lid}: {type(exc).__name__}: {exc}")
        continue
    if not v.rooms:
        continue
    back = onto_source([r.polygon_px for r in v.rooms], pi)
    ours[lid] = {"rooms": [{"polygon_px": p, "label": r.label or ""}
                           for p, r in zip(back, v.rooms)]}

score("(for reference) our old reading", ours,
      "our own software: wall model, rooms grown from the printed names")

---

# Part 4 — The report

### The table

In [ ]:
base = LADDER[0]
best = max(LADDER, key=lambda e: e["score"])

print(f'{"":3}{"reading":<44}{"score":>8}{"rooms found":>13}{"wall match":>12}{"rooms":>7}')
print("-" * 87)
for i, e in enumerate(LADDER):
    mark = " <--" if e is best else ""
    print(f'{i:<3}{e["name"][:43]:<44}{e["score"]:>8.0%}{e["found"]:>13.0%}'
          f'{e["edge"]:>12.0%}{e["rooms"]:>7}{mark}')
print("-" * 87)
print(f'\nbest: {best["name"]} — {best["score"]:.0%}, from {base["score"]:.0%} at the base')
print(f'({best["note"]})')

# Reading the two halves apart is the whole point of having two.
print(f'\n  rooms found  {base["found"]:.0%} -> {best["found"]:.0%}   '
      f'(share of the names printed on the plan that landed in exactly one room)')
print(f'  wall match   {base["edge"]:.0%} -> {best["edge"]:.0%}   '
      f'(share of each room edge that sits on a wall)')

print(f'\nwhat you kept: {CURRENT["name"]} ({CURRENT["score"]:.0%}), '
      f'grown to the walls: {GROWN["score"]:.0%}' if GROWN else "")

reference = next((e for e in LADDER if e["name"].startswith("(for reference)")), None)
if reference:
    verdict = "better than" if best["score"] > reference["score"] + 0.01 else (
        "no better than" if best["score"] > reference["score"] - 0.01 else "worse than")
    print(f'\n  This is {verdict} our old reading ({reference["score"]:.0%}).')

### The chart

In [ ]:
import matplotlib.pyplot as plt

BAR, REF = "#2a78d6", "#9aa0a6"
INK, MUTED, RULE = "#0b0b0b", "#52514e", "#d8d8d2"

shown = [e for e in LADDER]
names = [e["name"] for e in shown]
scores = [e["score"] for e in shown]
y = np.arange(len(shown))

fig, ax = plt.subplots(figsize=(11.5, 0.6 * len(shown) + 2.2))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")
ax.barh(y, scores, height=0.6, zorder=3,
        color=[REF if e["name"].startswith("(for reference)") else BAR for e in shown])
ax.axvline(scores[0], color=MUTED, linestyle="--", linewidth=1, zorder=2)
ax.text(scores[0], -0.8, "the base", color=MUTED, fontsize=9, ha="center", va="bottom")

for i, s in enumerate(scores):
    ax.text(s + 0.01, i, f"{s:.0%}" + ("   best" if shown[i] is best else ""),
            va="center", fontsize=10, color=INK,
            fontweight="bold" if shown[i] is best else "normal")

ax.set_yticks(y); ax.set_yticklabels(names, fontsize=10, color=INK)
ax.invert_yaxis()
ax.set_xlim(0, max(scores) * 1.25)
ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.set_xticklabels([f"{v:.0%}" for v in np.arange(0, 1.01, 0.2)], color=MUTED, fontsize=9)
ax.set_xlabel("rooms found x wall match", color=MUTED, fontsize=10)
ax.grid(axis="x", color=RULE, linewidth=1, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(RULE)
ax.tick_params(length=0)
ax.set_title(f"Every reading, on the same {base['plans']} plans", fontsize=13,
             color=INK, loc="left", pad=22)
plt.tight_layout(); plt.show()

### Every flat, end to end

| panel | what it is |
|---|---|
| 1 | the plan |
| 2 | the room-finder **as published** — the base |
| 3 | the room-finder **tuned** — the best row in the table |
| 4 | the same rooms with their **corners pulled onto the walls** |
| 5 | **our old reading**, for reference |

Read the shapes, not the numbers:

- are the **bathroom, WC, hallway and cupboards** there at all?
- is an **open-plan kitchen-and-living-room one room**, or two?
- do any rooms **shoot out past the building** into the margin? Nothing in the table can see
  that, so this is the only place it shows up.

In [ ]:
reference = next((e for e in LADDER if e["name"].startswith("(for reference)")), None)

compare([base, CURRENT, GROWN, reference],
        labels=["as published", f"what you kept: {CURRENT['name']}",
                "corners on the walls", "our old reading"])

### Save the results, and the recipe that produced them

In [ ]:
import shutil

OUT = ROOT / "results"
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()


def slug_of(name):
    return "_".join("".join(ch if ch.isalnum() else " " for ch in name.lower()).split())


manifest = {}
for e in LADDER:
    slug = slug_of(e["name"])
    (OUT / f"{slug}.json").write_text(json.dumps(e["reading"], indent=1))
    manifest[slug] = {
        "kind": ("ours" if e["name"].startswith("(for reference)")
                 else "combined" if e["name"].startswith("+ corners")
                 else "room_finder"),
        "name": e["name"], "score": e["score"],
        "found": e["found"], "edge": e["edge"], "rooms": e["rooms"]}
(OUT / "readings.json").write_text(json.dumps(manifest, indent=1))
# `reading` is the rooms, saved separately; `step` is the function that made the
# picture, which is not a thing JSON can hold.
NOT_JSON = {"reading", "step"}
(OUT / "every_reading.json").write_text(json.dumps(
    [{k: v for k, v in e.items() if k not in NOT_JSON} for e in LADDER], indent=1))

# The recipe, in a form you can act on.
print("THE RECIPE YOU KEPT")
print(f"  checkpoint      {CHECKPOINT}")
print(f"  picture         {' -> '.join(f.__name__ for f in RECIPE) or 'the plan as published'}")
print(f"  reading         {CURRENT['name']}")
print(f"  scored          {CURRENT['score']:.0%}  "
      f"(rooms found {CURRENT['found']:.0%}, wall match {CURRENT['edge']:.0%})")
if GROWN:
    print(f"  grown to walls  {GROWN['score']:.0%}  "
          f"(rooms found {GROWN['found']:.0%}, wall match {GROWN['edge']:.0%})")

pick = slug_of(CURRENT["name"])
print(f"\nimport it into the pipeline with:")
print(f"    python -m tools.import_room_predictions results.zip --reading {pick}")
print("(that is the room-finder's own rooms — the pipeline pulls the corners onto the")
print(" walls itself, which is what the last row of the table did.)")

archive = Path(shutil.make_archive(str(ROOT / "results"), "zip", OUT))
print(f"\n{archive}  ({archive.stat().st_size / 1e6:.1f} MB)")
if VOLUME is not None:
    try:
        Path(VOLUME).mkdir(parents=True, exist_ok=True)
        shutil.copy(archive, Path(VOLUME) / "results.zip")
        print(f"copied to {Path(VOLUME) / 'results.zip'} — survives this machine stopping")
    except Exception as exc:                                     # noqa: BLE001
        print(f"could not copy to {VOLUME}: {exc}")
else:
    print("Download it from the file browser in the sidebar, or set VOLUME at the top of")
    print("cell 1.2 to a mounted Volume and re-run this cell to keep it.")

---

## What to do with the answer

**If the tuned room-finder beat our old reading**, put its rooms into the pipeline:

```bash
python -m tools.import_room_predictions results.zip --reading <the name printed above>
python -m pipeline run <ids> --from 5
python -m tools.plan_vs_shell build --out out/review
```

**If a checkpoint other than `cubicasa5k` won**, that is the more important result — it means
the model we have been judging was never the right one for these plans.

**If the picture steps did most of the work**, they need to move into the pipeline too, so
that what stage 5 feeds the model matches what won here. They are all in this notebook's
`clean` / `crop` / `pure_lines` / `thicker` functions and none of them need a GPU.

**If nothing beat the base**, the room-finder is at its ceiling on our plans as it is, and the
next thing worth doing is teaching it our plans rather than tuning what it already does.